In [1]:
import tensorflow as tf 
import numpy as np

ds = tf.data.Dataset.range(10) # делаем датасет из чисел от 0 до 9
ds = ds.map(lambda x: x * 2) # применяем функцию к каждому элементу
ds = ds.batch(3) # собираем элементы в батчи по три штуки

for batch in ds.take(2): # берем первые 2 батча
    print(batch.numpy())

I0000 00:00:1793718087.551030    9110 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE3 SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


[0 2 4]
[ 6  8 10]


In [7]:
ds = tf.data.Dataset.range(10).shuffle(buffer_size=10).batch(5)

for epoch in range(2):
    print(f"Epoch: {epoch + 1}")
    for batch in ds:
        print(batch.numpy())

Epoch: 1
[6 5 8 0 3]
[1 2 9 7 4]
Epoch: 2
[7 9 3 6 5]
[1 4 8 0 2]


In [9]:
images = tf.random.uniform((12, 28, 28, 1), maxval=255, minval=1, dtype=tf.float32)
labels = tf.range(12) % 10

def preprocess(image, label):
    image = tf.image.resize(image, (32, 32)) # меняем размер изображения
    image = image / 255.0 # нормализуем пиксели 
    image = tf.image.random_flip_left_right(image) # случайное отражение
    return image, label

ds = (
    tf.data.Dataset.from_tensor_slices((images, labels))
    .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE) # позволяю тф самому выбрать колво параллельных операций
    .batch(4)
    .prefetch(tf.data.AUTOTUNE)
)

for x, y in ds.take(1):
    print(x.shape, y.numpy())

(4, 32, 32, 1) [0 1 2 3]


In [ ]:
dataset = (
    tf.data.Dataset.from_tensor_slices((X, y))
    .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .cache()
    .shuffle(buffer_size=1000)
    .batch(32)
    .prefetch(tf.data.AUTOTUNE)
)

In [11]:
def make_dataset(X, y, batch_size=32, training=False):
    ds = tf.data.Dataset.from_tensor_slices((X, y))

    def preprocess(x, label):
        x = tf.cast(x, tf.float32) / 255.0
        return x, label

    ds = ds.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)

    if training:
        ds = ds.cache()
        shuffle_buffer = min(10_000, len(X))
        ds = ds.shuffle(buffer_size=shuffle_buffer, reshuffle_each_iteration=True)
    else:
        ds = ds.cache()

    ds = ds.batch(batch_size)
    ds = ds.prefetch(tf.data.AUTOTUNE)

    return ds

In [12]:
X = np.random.randint(0, 256, size=(200, 28, 28, 1)).astype(np.float32)
y = np.random.randint(0, 10, size=(200,)).astype(np.int64)

train_ds = make_dataset(X[:160], y[:160], batch_size=16, training=True)
val_ds = make_dataset(X[160:], y[160:], batch_size=16, training=False)

In [13]:
model = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=(28, 28, 1)),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(10),
])

model.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy'],
)

/usr/lib/python3.14/site-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [14]:
history = model.fit(
    train_ds,
    epochs=2,
    validation_data=val_ds,
    verbose=2
)

Epoch 1/2


/usr/lib/python3.14/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


10/10 - 2s - 173ms/step - accuracy: 0.0938 - loss: 2.5094 - val_accuracy: 0.1000 - val_loss: 2.3946
Epoch 2/2
10/10 - 0s - 7ms/step - accuracy: 0.1000 - loss: 2.3248 - val_accuracy: 0.0500 - val_loss: 2.3656


In [17]:
ds = tf.data.Dataset.range(20)
ds = ds.map(lambda x: x  *2)
ds = ds.batch(5)

for batch in ds.take(2):
    print(batch.numpy())

[0 2 4 6 8]
[10 12 14 16 18]


In [4]:
dataset_1 = tf.data.Dataset.range(20).shuffle(buffer_size=5).batch(4)
dataset_2 = tf.data.Dataset.range(20).shuffle(buffer_size=20).batch(4)

for batch_1 in dataset_1:
    print(batch_1.numpy())

print('========================================')
for batch_2 in dataset_2:
    print(batch_2.numpy())

[2 3 4 1]
[6 8 7 9]
[12  0 14 10]
[15 13 11  5]
[17 19 18 16]
[ 9 12 17  1]
[11 15 18  6]
[7 3 2 4]
[13  5 14  0]
[16 10 19  8]


In [8]:
images = tf.random.uniform((16, 28, 28, 1), maxval=255, minval=1, dtype=tf.float32)
labels = tf.range(16) % 10

def image_preprocess(image, label):
    image = tf.cast(image, tf.float32) / 255.0
    image = tf.image.random_brightness(image, max_delta=0.2)
    image = tf.image.random_flip_left_right(image)
    return image, label

ds = (
    tf.data.Dataset.from_tensor_slices((images, labels))
    .map(image_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(4)
    .prefetch(tf.data.AUTOTUNE)
)

for x, y in ds.take(1):
    print(x.shape)
    print(y.numpy())

(4, 28, 28, 1)
[0 1 2 3]


In [10]:
model = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape = (28, 28, 1)),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(10),
])

model.compile(
    optimizer = 'adam',
    loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy'],
)

model.fit(ds, epochs=2, verbose = 2)

Epoch 1/2


/usr/lib/python3.14/site-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/usr/lib/python3.14/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


4/4 - 1s - 326ms/step - accuracy: 0.0625 - loss: 2.4868
Epoch 2/2
4/4 - 0s - 16ms/step - accuracy: 0.1875 - loss: 2.2658


In [3]:
# сжатие файлов

# запись
options = tf.io.TFRecordOptions(compression_type='GZIP')

with tf.io.TFRecordWriter('m_d.tfrecord.gz', options=options) as f:
    f.write(b"first rec")
    f.write(b"second rec")

In [5]:
dataset = tf.data.TFRecordDataset(
    ['m_d.tfrecord.gz'],
    compression_type='GZIP'
)

for record in dataset:
    print(record.numpy())

b'first rec'
b'second rec'


In [6]:
from tensorflow.train import Example, Features, Feature, BytesList, Int64List, FloatList

person_example = Example(
    features=Features(
        feature={
            "name": Feature(
                bytes_list=BytesList(value=[b"Aurelien Geron"])
            ),
            "id": Feature(
                int64_list=Int64List(value=[2345222])
            ),
            "scores": Feature(
                float_list=FloatList(value=[95.5, 88.0, 92.3])
            ),
            "emails": Feature(
                bytes_list=BytesList(value=[b"a@example.com", b"b@example.com"])
            ),
        }
    )
)

print(person_example)

features {
  feature {
    key: "scores"
    value {
      float_list {
        value: 95.5
        value: 88
        value: 92.3
      }
    }
  }
  feature {
    key: "name"
    value {
      bytes_list {
        value: "Aurelien Geron"
      }
    }
  }
  feature {
    key: "id"
    value {
      int64_list {
        value: 2345222
      }
    }
  }
  feature {
    key: "emails"
    value {
      bytes_list {
        value: "a@example.com"
        value: "b@example.com"
      }
    }
  }
}



In [7]:
with tf.io.TFRecordWriter("people.tfrecord") as f:
    f.write(person_example.SerializeToString())

In [ ]:
# чтение обратно
feature_description = {
    "name": tf.io.FixedLenFeature([], tf.string, default_value=""),
    "id": tf.io.FixedLenFeature([], tf.int64, default_value=0),
    "scores": tf.io.VarLenFeature(tf.float32),
    "emails": tf.io.VarLenFeature(tf.string),
}

for raw_record in tf.data.TFRecordDataset(["people.tfrecord"]):
    example = tf.io.parse_single_example(raw_record, feature_description)
    print(example["name"].numpy())
    print(example["id"].numpy())
    print(example["scores"].values.numpy())

b'Aurelien Geron'
2345222
[95.5 88.  92.3]


In [9]:
from tensorflow.train import (
    SequenceExample, Features, FeatureLists, FeatureList,
    Feature, Int64List
)

context = Features(
    feature={
        "movie_id": Feature(int64_list=Int64List(value=[12345])),
        "title": Feature(bytes_list=BytesList(value=[b"Inception"])),
    }
)

ratings = FeatureList(
    feature=[
        Feature(int64_list=Int64List(value=[5])),
        Feature(int64_list=Int64List(value=[3])),
        Feature(int64_list=Int64List(value=[4])),
        Feature(int64_list=Int64List(value=[5])),
    ]
)

seq_example = SequenceExample(
    context=context,
    feature_lists=FeatureLists(feature_list={"ratings": ratings})
)


with tf.io.TFRecordWriter("movies.tfrecord") as f:
    f.write(seq_example.SerializeToString())

In [10]:
context_features = {
    "movie_id": tf.io.FixedLenFeature([], tf.int64),
    "title": tf.io.FixedLenFeature([], tf.string),
}

sequence_features = {
    "ratings": tf.io.FixedLenSequenceFeature([], tf.int64),
}

for raw_record in tf.data.TFRecordDataset(["movies.tfrecord"]):
    context_parsed, sequence_parsed = tf.io.parse_single_sequence_example(
        raw_record,
        context_features=context_features,
        sequence_features=sequence_features,
    )
    print("Movie:", context_parsed["title"].numpy())
    print("Ratings:", sequence_parsed["ratings"].numpy())

Movie: b'Inception'
Ratings: [5 3 4 5]


In [2]:
num_samples = 100
image_size = 28

images = np.random.rand(num_samples, image_size, image_size, 1).astype(np.float32)
labels = np.random.randint(0, 10, size=(num_samples,)).astype(np.int64)

print(f"Создано {num_samples} изображений размером {image_size}x{image_size}")

def image_example(image, label):
    image_raw = image.tobytes()  # превращаем массив в байты

    example = tf.train.Example(
        features=tf.train.Features(
            feature={
                "image_raw": tf.train.Feature(
                    bytes_list=tf.train.BytesList(value=[image_raw])
                ),
                "height": tf.train.Feature(
                    int64_list=tf.train.Int64List(value=[image_size])
                ),
                "width": tf.train.Feature(
                    int64_list=tf.train.Int64List(value=[image_size])
                ),
                "channels": tf.train.Feature(
                    int64_list=tf.train.Int64List(value=[1])
                ),
                "label": tf.train.Feature(
                    int64_list=tf.train.Int64List(value=[label])
                ),
            }
        )
    )
    return example

tfrecord_file = "images.tfrecord"

with tf.io.TFRecordWriter(tfrecord_file) as writer:
    for i in range(num_samples):
        example = image_example(images[i], labels[i])
        writer.write(example.SerializeToString())

print(f"Записано {num_samples} примеров в {tfrecord_file}")


feature_description = {
    "image_raw": tf.io.FixedLenFeature([], tf.string),
    "height": tf.io.FixedLenFeature([], tf.int64),
    "width": tf.io.FixedLenFeature([], tf.int64),
    "channels": tf.io.FixedLenFeature([], tf.int64),
    "label": tf.io.FixedLenFeature([], tf.int64),
}

def parse_function(raw_record):
    parsed = tf.io.parse_single_example(raw_record, feature_description)

    image = tf.io.decode_raw(parsed["image_raw"], tf.float32)
    image = tf.reshape(image, [parsed["height"], parsed["width"], parsed["channels"]])

    label = parsed["label"]
    return image, label

dataset = tf.data.TFRecordDataset([tfrecord_file])
dataset = dataset.map(parse_function, num_parallel_calls=tf.data.AUTOTUNE)
dataset = dataset.shuffle(buffer_size=100)
dataset = dataset.batch(16)
dataset = dataset.prefetch(tf.data.AUTOTUNE)

for images_batch, labels_batch in dataset.take(1):
    print(f"Форма батча изображений: {images_batch.shape}")
    print(f"Форма батча меток: {labels_batch.shape}")
    print(f"Метки: {labels_batch.numpy()}")


model = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=(28, 28, 1)),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(10),
])

model.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy'],
)

model.fit(dataset, epochs=2, verbose=2)

print("Все готово!")

Создано 100 изображений размером 28x28
Записано 100 примеров в images.tfrecord
Форма батча изображений: (16, 28, 28, 1)
Форма батча меток: (16,)
Метки: [0 9 0 4 3 6 7 6 5 2 4 9 5 3 0 5]
Epoch 1/2


I0000 00:00:1793718278.987997    9182 tf_record_dataset_op.cc:390] TFRecordDataset `buffer_size` is unspecified, default to 262144
/usr/lib/python3.14/site-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/usr/lib/python3.14/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


7/7 - 1s - 207ms/step - accuracy: 0.1000 - loss: 2.4690
Epoch 2/2
7/7 - 0s - 10ms/step - accuracy: 0.1500 - loss: 2.2695
Все готово!


/usr/lib/python3.14/site-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


In [6]:
file_name = 'numbers.tfrecord'

with tf.io.TFRecordWriter(file_name) as file:
    for i in range(5):
        file.write(f'record_{i}'.encode())

dataset =  tf.data.TFRecordDataset([file_name])

for s in dataset:
    print(s.numpy().decode())

record_0
record_1
record_2
record_3
record_4


In [8]:
options = tf.io.TFRecordOptions(compression_type="GZIP")

with tf.io.TFRecordWriter("numbers_compressed.tfrecord.gz", options=options) as f:
    for i in range(5):
        f.write(f"record_{i}".encode())

dataset = tf.data.TFRecordDataset(
    ["numbers_compressed.tfrecord.gz"],
    compression_type="GZIP"
)
for record in dataset:
    print(record.numpy().decode())

record_0
record_1
record_2
record_3
record_4


In [ ]:
# One hot encoding
cities = ["Москва", "Санкт-Петербург", "Новосибирск", "Москва", "Казань"]

vocab = ["Москва", "Санкт-Петербург", "Новосибирск", "Казань"]

cities_indices = [vocab.index(city) for city in cities]
print("Индексы:", cities_indices)

one_hot = tf.one_hot(cities_indices, depth=len(vocab))
print("\nOne-hot векторы:")
for i, city in enumerate(cities):
    print(f"{city}: {one_hot[i].numpy()}")

Индексы: [0, 1, 2, 0, 3]

One-hot векторы:
Москва: [1. 0. 0. 0.]
Санкт-Петербург: [0. 1. 0. 0.]
Новосибирск: [0. 0. 1. 0.]
Москва: [1. 0. 0. 0.]
Казань: [0. 0. 0. 1.]


In [ ]:
cities_indices = np.array([0, 1, 2, 0, 3])

num_categories = 4 
embedding_dim = 10 

embedding_layer = tf.keras.layers.Embedding(
    input_dim=num_categories,
    output_dim=embedding_dim
)

embedded_cities = embedding_layer(cities_indices)

print("Размерность одного embedding:", embedded_cities[0].shape)
print("\nEmbedding для 'Москва' (индекс 0):")
print(embedded_cities[0].numpy())
print("\nEmbedding для 'Санкт-Петербург' (индекс 1):")
print(embedded_cities[1].numpy())

Размерность одного embedding: (10,)

Embedding для 'Москва' (индекс 0):
[ 0.03529737 -0.01215327 -0.04544927  0.02054442 -0.01290113 -0.03809886
 -0.04925799  0.03999623 -0.03872075 -0.04379928]

Embedding для 'Санкт-Петербург' (индекс 1):
[-0.0375801  -0.00121895  0.00419516  0.02752943  0.01325306 -0.041329
 -0.01478255 -0.04355163  0.01754311 -0.0394026 ]


In [12]:
cities = tf.constant(["Москва", "Санкт-Петербург", "Новосибирск", "Москва", "Казань"])

lookup_layer = tf.keras.layers.StringLookup()

lookup_layer.adapt(cities)

city_indices = lookup_layer(cities)
print("Индексы:", city_indices.numpy())

print("\nСловарь:", lookup_layer.get_vocabulary())

Индексы: [1 2 3 1 4]

Словарь: ['[UNK]', np.str_('Москва'), np.str_('Санкт-Петербург'), np.str_('Новосибирск'), np.str_('Казань')]


In [13]:
city_indices = tf.constant([2, 4, 3, 2, 1]) 

num_categories = 5 
one_hot_layer = tf.keras.layers.CategoryEncoding(
    num_tokens=num_categories,
    output_mode="one_hot"
)

one_hot_cities = one_hot_layer(city_indices)
print("One-hot:")
print(one_hot_cities.numpy())

One-hot:
[[0. 0. 1. 0. 0.]
 [0. 0. 0. 0. 1.]
 [0. 0. 0. 1. 0.]
 [0. 0. 1. 0. 0.]
 [0. 1. 0. 0. 0.]]


In [14]:
hashing_layer = tf.keras.layers.Hashing(num_bins=100)

hashed_cities = hashing_layer(cities)
print("Хешированные значения:", hashed_cities.numpy())

Хешированные значения: [69 44  1 69 71]


In [15]:
ages = tf.constant([18, 25, 35, 45, 60, 75])

bin_boundaries = [0, 18, 30, 50, 100]

discretize_layer = tf.keras.layers.Discretization(bin_boundaries=bin_boundaries)

binned_ages = discretize_layer(ages)
print("Биннированные возраста:", binned_ages.numpy())


Биннированные возраста: [2 2 3 3 4 4]


In [18]:
np.random.seed(42)
n_samples = 100

age = np.random.randint(18, 80, size=(n_samples,)).astype(np.float32).reshape(-1, 1)
income = np.random.randint(20000, 150000, size=(n_samples,)).astype(np.float32).reshape(-1, 1)

cities = np.random.choice(
    ["Москва", "Санкт-Петербург", "Новосибирск", "Казань", "Екатеринбург"],
    size=(n_samples,)
).reshape(-1, 1) 

y = np.random.randint(0, 2, size=(n_samples,))

dataset = tf.data.Dataset.from_tensor_slices((
    {"age": age, "income": income, "city": cities},
    y
))

age_layer = tf.keras.layers.Normalization()
age_layer.adapt(tf.constant(age))

income_layer = tf.keras.layers.Normalization()
income_layer.adapt(tf.constant(income))

city_lookup = tf.keras.layers.StringLookup()
city_lookup.adapt(tf.constant(cities))

city_one_hot = tf.keras.layers.CategoryEncoding(
    num_tokens=city_lookup.vocabulary_size(),
    output_mode="one_hot"
)

inputs = {
    "age": tf.keras.Input(shape=(1,), dtype=tf.float32),
    "income": tf.keras.Input(shape=(1,), dtype=tf.float32),
    "city": tf.keras.Input(shape=(1,), dtype=tf.string),
}

age_normalized = age_layer(inputs["age"])
income_normalized = income_layer(inputs["income"])
city_indices = city_lookup(inputs["city"])
city_encoded = city_one_hot(city_indices)

concatenated = tf.keras.layers.Concatenate()([
    age_normalized,
    income_normalized,
    city_encoded
])

x = tf.keras.layers.Dense(32, activation="relu")(concatenated)
x = tf.keras.layers.Dense(16, activation="relu")(x)
outputs = tf.keras.layers.Dense(1, activation="sigmoid")(x)

model = tf.keras.Model(inputs=inputs, outputs=outputs)

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

dataset = dataset.batch(16)

history = model.fit(dataset, epochs=3, verbose=2)

print("\nМодель обучена!")
print("\nПопробуем предсказать:")
test_input = {
    "age": tf.constant([[30.0]]),
    "income": tf.constant([[50000.0]]),
    "city": tf.constant([["Москва"]]),
}
prediction = model.predict(test_input, verbose=0)
print(f"Предсказание: {prediction[0][0]:.4f}")

Epoch 1/3


/usr/lib/python3.14/site-packages/keras/src/models/functional.py:106: UserWarning: When providing `inputs` as a dict, all keys in the dict must match the names of the corresponding tensors. Received key 'age' mapping to value <KerasTensor shape=(None, 1), dtype=float32, sparse=False, ragged=False, name=keras_tensor_26> which has name 'keras_tensor_26'. Change the tensor name to 'age' (via `Input(..., name='age')`)
  warnings.warn(
/usr/lib/python3.14/site-packages/keras/src/models/functional.py:106: UserWarning: When providing `inputs` as a dict, all keys in the dict must match the names of the corresponding tensors. Received key 'income' mapping to value <KerasTensor shape=(None, 1), dtype=float32, sparse=False, ragged=False, name=keras_tensor_27> which has name 'keras_tensor_27'. Change the tensor name to 'income' (via `Input(..., name='income')`)
  warnings.warn(
/usr/lib/python3.14/site-packages/keras/src/models/functional.py:106: UserWarning: When providing `inputs` as a dict, all

7/7 - 2s - 286ms/step - accuracy: 0.4400 - loss: 0.6959
Epoch 2/3
7/7 - 0s - 4ms/step - accuracy: 0.5600 - loss: 0.6890
Epoch 3/3
7/7 - 0s - 4ms/step - accuracy: 0.5900 - loss: 0.6846

Модель обучена!

Попробуем предсказать:
Предсказание: 0.5330
